In [8]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np

# 1. Dummy Data (Thoda bada, taaki model seekh sake)
# Real world mein hum IMDB dataset use karte hain (~50,000 reviews)

texts = [
    "I loved this movie, it was amazing", "Terrible film, worst acting ever",
    "A masterpiece, brilliant direction", "Boring and waste of time",
    "Excellent story, great performances", "I hated it, very disappointing",
    "Fantastic, would watch again", "Not good, poor screenplay",
    "Superb acting, loved every bit", "Horrible, don't watch"
]

labels = [1, 0, 1, 0, 1, 0, 1, 0, 1, 0] # 1 = Positive, 0 = Negative

# Simple Vocabulary banate hain
all_words = " ".join(texts).lower().split()
vocab = {word: i+1 for i, word in enumerate(set(all_words))} # 0 is reserved for padding
vocab_size = len(vocab) + 1

# Text ko numbers mein badlo
def encode(text):
    return [vocab.get(w, 0) for w in text.lower().split()]

encoded_texts = [encode(t) for t in texts]

# Padding Function (sabhi sentences ko same length ka banana)
max_len = 10
def pad_sequence(seq, max_len):
    if len(seq) < max_len:
        return seq + [0] * (max_len - len(seq))
    return seq[:max_len]

padded_texts = [pad_sequence(t, max_len) for t in encoded_texts]

# Dataset Class
class SentimentDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        return torch.tensor(self.texts[idx], dtype=torch.long), torch.tensor(self.labels[idx], dtype=torch.float32)

dataset = SentimentDataset(padded_texts, labels)
loader = DataLoader(dataset, batch_size=2, shuffle=True)

print("Vocab Size:", vocab_size)
print("Sample Text Encoded:", padded_texts[0])

Vocab Size: 44
Sample Text Encoded: [4, 41, 23, 30, 26, 28, 37, 0, 0, 0]


In [11]:
class SentimentRNN(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size, output_size):
        super(SentimentRNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.rnn = nn.RNN(embed_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        embed = self.embedding(x)
        out, hidden = self.rnn(embed)
        # Last hidden state nikaalo
        out = self.fc(hidden.squeeze(0))
        out = self.sigmoid(out)
        return out

model = SentimentRNN(vocab_size=vocab_size, embed_size=8, hidden_size=16, output_size=1)
print(model)

SentimentRNN(
  (embedding): Embedding(44, 8)
  (rnn): RNN(8, 16, batch_first=True)
  (fc): Linear(in_features=16, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)


In [12]:
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

epochs = 20
print("Training shuru...")

for epoch in range(epochs):
    total_loss = 0
    for text, label in loader:
        optimizer.zero_grad()
        output = model(text).squeeze()
        loss = criterion(output, label)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    if (epoch+1) % 5 == 0:
        print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(loader):.4f}")
print("Training Complete!")

Training shuru...
Epoch 5/20 | Loss: 0.3945
Epoch 10/20 | Loss: 0.0534
Epoch 15/20 | Loss: 0.0133
Epoch 20/20 | Loss: 0.0063
Training Complete!
